# Lab 07: Artificial Neural Networks - Predicting Bike Rentals with Weather Data

## Part 1: ANN Concepts

### Task 1: Conceptual understanding

**What is an Artificial Neural Network?**

A computational model inspired by the human brain. It consists of interconnected layers of nodes (neurons) that process information, learn patterns from the data, and make decisions.

**What is a neuron?**

The building block of ANN. Receives numerical inputs, mathematically transforms them using weights nad bias, applies activation function, passes the results to the next layer of neurons.

**Explain the concepts of weights, bias, and activation function.**

* **Weights:** Learnable values that determine the importance of the connection between two neurons. They multiply the input data to increase or decrease its influence on the output.
* **Bias:** Additional, learnable constant added to the weighted sum of the inputs before the activation function is applied. It allows the activation function to be shifted left or right, helping the model fit the data more flexibly.
* **Activation function:** Mathematical formula applied to a neuron's output. It introduces non-linearity into the network, allowing the model to learn and represent complex, non-linear real-world patterns.

**What is forward propagation?**

The directional process of passing data through the neural network. The input data moves forward from the input layer, through any hidden layers, to the output layer to generate a final prediction.

**What is a loss function?**

A mathematical method used to measure how far off the neural network's prediction is from the actual, true target value. Quantifies the error of the model. The goal during model training is to minimize this loss.

**What is backpropagration?**

Process of learning from mistakes. After forward propagation calculates the error through the loss function, backpropagation works backward through the network's layers to calculate how much each weight and bias contribute to the error, updating them to improve future predictions.

### Task 2: Mapping concepts to the problem

**Input layer**

Raw data used to make predictions such as
* Temperature
* Humidity
* Wind speed

**Weights**

How much influence each weather condition has on hte final number of bike rentals. Wind speed might be a massive driver of whether people rent bikes, the network will learn to assign a higher weight to the wind speed input compared to the others.

**Output**

Final result generated by the network. In this case the final output will be a single numerical value which represents the number of bike rentals.

**Loss function**

Calculates the penalty for getting the prediction wrong. Measures the difference between the network's predicted number of bike rentals and the actual, historical number of bike rentals.

## Part 2: Data Collection (API)

### Task 3: Retrieve weather data

In [1]:
import requests

url = "https://api.open-meteo.com/v1/forecast?latitude=38.72&longitude=-9.14&current_weather=true"

data = requests.get(url).json()

temp = data["current_weather"]["temperature"]
wind = data["current_weather"]["windspeed"]
humidity = 60 # simplified

print(f"Temperature: {temp}°C, Wind Speed: {wind} km/h, Humidity: {humidity}%")

Temperature: 15.3°C, Wind Speed: 14.1 km/h, Humidity: 60%


### Task 4: Interpretation

**How does temperature affect bike rentals?**

Generally, temperature has a non-linear (often inverted U-shape) relationship with bike rentals. As temperatures rise from cold to mild/warm, bike rentals typically increase because the weather is more comfortable for outdoor activities. However, once the temperature gets excessively hot, rentals will usually drop off again as it becomes physically exhausting or dangerous to ride.

**How does wind influence demand?**

Wind usually has a negative correlation with bike rentals. Higher wind speeds make pedaling much more difficult and riding less enjoyable (and potentially more dangerous). Therefore, as wind speed increases, you can expect bike rental demand to decrease.

## Part 3: Dataset construction

### Task 5: Analyze dataset

In [2]:
dataset = [
[10, 80, 15, 100],
[15, 70, 10, 150],
[20, 60, 8, 220],
[25, 50, 5, 300],
[30, 40, 3, 280],
[12, 85, 20, 90]
]

**Why this dataset is realistic**

Temperatures range from 10°C to 30°C, humidity fluctuates between 40% and 85%, and wind speeds vary from 3 km/h to 20 km/h. The number of rentals, ranging from 90 to 300, reflects human behavior changing in response to those weather conditions.

**What patterns you observe**

* As the temperature rises from 10°C to 25°C, bike rentals climb from 100 to 300. However, when the temperature hits 30°C, rentals dip to 280, showing the U shape curve previously explained.
* The lowest rental days occr on the days with the highest wind speeds and highest humidity.

## Part 4: PyTorch Implementation

### Task 6: Build the model

In [3]:
import torch
import torch.nn as nn

data = torch.tensor(dataset, dtype=torch.float32)
X = data[:, :3]
y = data[:, 3].unsqueeze(1)

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.l1 = nn.Linear(3, 8)
        self.l2 = nn.Linear(8, 1)

    def forward(self, x):
        x = torch.relu(self.l1(x))
        x = self.l2(x)
        return x

model = Model()

### Task 7: Train the model

In [4]:
import torch.optim as optim

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(2000):
    pred = model(X)
    loss = criterion(pred, y)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

### Task 8: Make predictions

In [5]:
new_data = torch.tensor([[temp, humidity, wind]])
print(model(new_data))
print("Predicted rentals:", model(new_data).item())

tensor([[125.0594]], grad_fn=<AddmmBackward0>)
Predicted rentals: 125.05940246582031


### Task 9: Reflection

**Where are the weights?**

The weights (and biases) are automatically created, initialized, and stored inside the nn.Linear layers within the __init__ method. Specifically, nn.Linear(3, 8) creates the weights connecting the 3 input features to the 8 hidden neurons.

**Where does forward propagation occur?**

The logic for forward propagation is defined inside the def forward(self, x): method of the class. It is actively executed during the training loop when we call the model to make a prediction on the data: pred = model(X).

**Where is backpropagation implemented?**

Backpropagation occurs in the training loop using two specific lines:
* loss.backward() computes the gradients (the direction and magnitude of the error) by working backward through the network.
* optimizer.step() then applies those gradients to update the actual weights and biases, completing the learning step.

## Part 5: TensorFlow Implementation

### Task 10: Build and train model

In [8]:
import tensorflow as tf
import numpy as np

data = np.array(dataset)

X = data[:, :3]
y = data[:, 3]

model = tf.keras.Sequential([
    tf.keras.layers.Dense(8, activation='relu', input_shape=(3,)),
    tf.keras.layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse')

model.fit(X, y, epochs=2000, verbose=0)

### Task 11: Predict

In [11]:
# Convert the Python list into a NumPy array first
new_data = np.array([[temp, humidity, wind]])

# Now make the prediction
prediction = model.predict(new_data)
print(f"Predicted rentals: {prediction[0][0]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
Predicted rentals: 152.76498413085938


## Part 6: Business Interpretation

### Task 12: Decision-making

**If predicted rentals = 300, what should the company do?**

* **Proactive Rebalancing:** Move bikes from low-traffic residential areas to high-traffic commercial or tourist hubs before the rush begins to optimize bike distribution.

* **Halt Maintenance:** Delay any non-essential repairs or maintenance so that the maximum number of bikes are available on the street.

* **Dynamic Pricing:** If the business model allows, slightly increase rental rates during this peak window to maximize revenue.

**If predicted rentals = 90, what actions should be taken?**

* **Schedule Maintenance:** Use this downtime to pull bikes off the street for major repairs, battery charging (if e-bikes), and general fleet maintenance. This reduces operational costs by avoiding paying staff for idle time.

* **Run Promotions:** Send out push notifications with discount codes or lowered rates to incentivize the few people who are willing to brave the weather to choose your bikes over competitors.

* **Reallocate Staff:** Shift ground team members from rebalancing duties to warehouse repair duties.

**What are the limitations of this model?**

* **Missing Features:** It only looks at temperature, humidity, and wind speed. It completely ignores massive demand drivers like precipitation (rain/snow), time of day (rush hour vs. 3 AM), day of the week (weekends vs. weekdays), and holidays.
* **Tiny Dataset:** The training dataset only has six examples. A real-world neural network needs thousands or millions of data points to generalize well. Training a model on six rows just forces it to memorize those specific numbers (overfitting).
* **Simplified Architecture:** A small network with a single hidden layer of 8 neurons is unlikely to capture the complex, non-linear relationships of human behavior.